# 02 — Figure 3 | GC Content, CAI, and Phylogenetic Distance Analysis

## Configuration

In [ ]:
from pathlib import Path

# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- paths ---
GENE_DATA_PATH = DATA_DIR / "../data/gene_fitness_results_with_annotations.parquet"

# --- analysis parameters ---
CONDITION = "LB_4_salt"


## Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
import scipy.stats as stats


df_raw = pd.read_parquet(GENE_DATA_PATH)
print(f"{len(df_raw):,} rows loaded")

## Data preparation

In [ ]:
# --- filter to salt condition genes with a gene call ---
salt_genes = df_raw[df_raw["condition"] == CONDITION].copy()

# --- patristic distances (written by figure1.ipynb to results/) ---
phylogeny = pd.read_csv(RESULTS_DIR / "../results/pioneer_genome_gtdb_phylogeny.tsv", sep="\t")
distance_to_ecoli = phylogeny[["pioneer_genome", "distance_to_ecoli"]]

# NOTE: Sapiens_RBS_Score is already merged into the released gene-fitness table,
# so no separate RBS merge step is needed here (unlike the original internal notebook).
print(salt_genes.shape)
salt_genes.head()


In [ ]:
salt_genes_filt = salt_genes[
    salt_genes["gene_call"].isin(["Hit", "Not Hit"])
].copy()

print(f"{len(salt_genes_filt):,} genes with Hit/Not Hit call")

salt_genes_filt.head()

## Figure 3a — Fitness effects of hits by phylogenetic distance to E. coli

### Generate species level summary metrics

In [ ]:
from adjustText import adjust_text
import seaborn as sns

### Genome hit counts and effect magnitude
per_genome_hit_info = (
    salt_genes_filt[(salt_genes_filt["gene_call"] == "Hit")]
    .groupby(["pioneer_genome", "species_name"])
    .agg(
        n_hits=(        "lmer_estimate", lambda x: np.sum(x > 0)),
        avg_magnitude=( "lmer_estimate", lambda x: np.abs(x[x>0]).mean()),
        max_magnitude=( "lmer_estimate", "max"),
    )
    .reset_index()
    .merge(distance_to_ecoli, on="pioneer_genome", how="left")
)
per_genome_hit_info = per_genome_hit_info.merge(salt_genes_filt.groupby(["pioneer_genome", "species_name"]).agg(pass_QC_gene_count = ("gene_id", "count")).reset_index(),
                                            on = ["pioneer_genome", "species_name"], how = "left")

per_genome_hit_info["hit_rate_per1k"] = per_genome_hit_info["n_hits"] / per_genome_hit_info["pass_QC_gene_count"] * 1000

### All gene metrics
median_e_coli_gc = salt_genes[salt_genes["species_name"] == "Escherichia coli"]["gc_content"].median()

genome_metrics_info = salt_genes.groupby("species_name").agg(median_gc=("gc_content", "median"),
                                                    median_CAI=("CAI_rel_to_Ecoli", "median"),
                                                    median_RBS_score=("Sapiens_RBS_Score", "median")).reset_index()

genome_metrics_info["median_gc_diff"] = np.abs(genome_metrics_info["median_gc"] - median_e_coli_gc)
genome_metrics_info.sort_values("median_gc_diff", ascending=False)

per_genome_hit_info = per_genome_hit_info.merge(genome_metrics_info, on="species_name", how="left")
per_genome_hit_info.sort_values("median_gc_diff", ascending=False)

Phylogenetic Distance

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sc = ax.scatter(
    per_genome_hit_info["distance_to_ecoli"],
    per_genome_hit_info["hit_rate_per1k"],
    c=per_genome_hit_info["max_magnitude"],
    s=80,
    cmap="Reds",
)
fig.colorbar(sc, ax=ax, label="Max sig. positive\nfitness effect", shrink=0.5, location="right")

texts = [
    ax.text(
        row["distance_to_ecoli"],
        row["hit_rate_per1k"],
        f"{row['species_name'].split()[0][0]}. {row['species_name'].split()[1]}",
        fontsize=8,
    )
    for _, row in per_genome_hit_info.iterrows()
]

ax.set_xlabel("Phylogenetic distance", fontsize=10)
ax.set_ylabel("Rate of sig pos. genes per 1K", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
adjust_text(texts, ax=ax, expand_objects=(1.5, 1.5),)

fig.savefig(RESULTS_DIR / "figure3a_fitness_effects_by_phylo_distance.pdf", bbox_inches="tight")

plt.show()


In [ ]:
smf.ols("hit_rate_per1k ~ distance_to_ecoli", data=per_genome_hit_info).fit().summary()

## Figure 3b — Fitness effects of hits by GC content distance to E. coli

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sc = ax.scatter(
    per_genome_hit_info["median_gc_diff"],
    per_genome_hit_info["hit_rate_per1k"],
    c=per_genome_hit_info["max_magnitude"],
    s=80,
    cmap="Reds",
)
fig.colorbar(sc, ax=ax, label="Max sig. positive\nfitness effect", shrink=0.5, location="right")

texts = [
    ax.text(
        row["median_gc_diff"],
        row["hit_rate_per1k"],
        f"{row['species_name'].split()[0][0]}. {row['species_name'].split()[1]}",
        fontsize=8,
    )
    for _, row in per_genome_hit_info.iterrows()
]
adjust_text(texts, ax=ax, expand_points=(2, 2), expand_text=(2, 2))

ax.set_xlabel("Absolute median GC content difference", fontsize=10)
ax.set_ylabel("Rate of sig pos. genes per 1K", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

plt.show()

fig.savefig(RESULTS_DIR / "figure3b_fitness_effects_by_gc_distance.pdf", bbox_inches="tight")


In [ ]:
smf.ols("hit_rate_per1k ~ median_gc_diff", data=per_genome_hit_info).fit().summary()

In [ ]:
smf.ols("hit_rate_per1k ~ median_gc_diff + distance_to_ecoli", data=per_genome_hit_info).fit().summary()

## Figure 3c — Fitness effects of hits by RBS binding score

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sc = ax.scatter(
    per_genome_hit_info["median_RBS_score"],
    per_genome_hit_info["hit_rate_per1k"],
    c=per_genome_hit_info["max_magnitude"],
    s=80,
    cmap="Reds",
)
fig.colorbar(sc, ax=ax, label="Max sig. positive\nfitness effect", shrink=0.5, location="right")

texts = [
    ax.text(
        row["median_RBS_score"],
        row["hit_rate_per1k"],
        f"{row['species_name'].split()[0][0]}. {row['species_name'].split()[1]}",
        fontsize=8,
    )
    for _, row in per_genome_hit_info.iterrows()
]
adjust_text(texts, ax=ax, expand_points=(2, 2), expand_text=(2, 2))

ax.set_xlabel("Median RBS score", fontsize=10)
ax.set_ylabel("Rate of sig pos. genes per 1K", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR / "figure3c_fitness_effects_by_rbs.pdf", bbox_inches="tight")


In [ ]:
smf.ols("hit_rate_per1k ~ median_RBS_score", data=per_genome_hit_info).fit().summary()

In [ ]:
smf.ols("hit_rate_per1k ~ median_RBS_score + distance_to_ecoli", data=per_genome_hit_info).fit().summary()

## Figure 3d — Fitness effects of hits by CAI relative to donor (E. coli)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

sc = ax.scatter(
    per_genome_hit_info["median_CAI"],
    per_genome_hit_info["hit_rate_per1k"],
    c=per_genome_hit_info["max_magnitude"],
    s=80,
    cmap="Reds",
)
fig.colorbar(sc, ax=ax, label="Max sig. positive\nfitness effect", shrink=0.5, location="right")

texts = [
    ax.text(
        row["median_CAI"],
        row["hit_rate_per1k"],
        f"{row['species_name'].split()[0][0]}. {row['species_name'].split()[1]}",
        fontsize=8,
    )
    for _, row in per_genome_hit_info.iterrows()
]
adjust_text(texts, ax=ax, expand_points=(2, 2), expand_text=(2, 2))

ax.set_xlabel("Median CAI with respect to donor (E. coli)", fontsize=10)
ax.set_ylabel("Rate of sig pos. genes per 1K", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR / "figure3d_fitness_effects_by_cai.pdf", bbox_inches="tight")


In [ ]:
smf.ols("hit_rate_per1k ~ median_CAI", data=per_genome_hit_info).fit().summary()

In [ ]:
smf.ols("hit_rate_per1k ~ median_CAI + distance_to_ecoli", data=per_genome_hit_info).fit().summary()

In [ ]:
smf.ols("hit_rate_per1k ~ median_gc_diff + median_RBS_score", data=per_genome_hit_info).fit().summary()

## Per Gene Hit Probability

### Data preparation

We work from `salt_genes_filt` (protein-coding, LB + 4% NaCl, Hit or Not Hit). The `gc_content` column is already present as a percentage (0–100 scale). We also pre-compute binned empirical hit probabilities (3% bins) which will be overlaid on the fitted curve as a visual sanity check.

### Statistical analysis

The empirical binning suggests a non-monotonic (inverted-U) relationship: hit probability is highest in the mid-GC range and drops toward both extremes. A standard logistic regression (sigmoid) would miss this, so we fit a **quadratic logistic regression**:

$$\log\frac{P(\text{Hit})}{1 - P(\text{Hit})} = \beta_0 + \beta_1 \cdot \text{GC} + \beta_2 \cdot \text{GC}^2$$

A significant negative $\beta_2$ confirms the inverted-U shape. We also run a **Mann–Whitney U test** as a non-parametric check that GC content distributions differ between Hits and Not Hits, without assuming any particular functional form.

In [ ]:
per_gene_df = (
    salt_genes_filt[["gene_id", "species_name", "gene_call", "gc_content", "CAI_rel_to_Ecoli", "Sapiens_RBS_Score", "lmer_estimate"]]
    .copy()
)
per_gene_df["is_hit"] = ((per_gene_df["gene_call"] == "Hit") & (per_gene_df["lmer_estimate"] > 0)).astype(int)
per_gene_df["gc_content_diff_ecoli"] = per_gene_df["gc_content"] - median_e_coli_gc

per_genome_hit_info["short_name"] = per_genome_hit_info["species_name"].apply(lambda x: f"{x.split()[0][0]}. {x.split()[1]}")
per_genome_hit_info.sort_values("distance_to_ecoli", inplace = True)

per_gene_df["short_name"] = per_gene_df["species_name"].apply(lambda x: f"{x.split()[0][0]}. {x.split()[1]}")

gc_df = per_gene_df.dropna(subset=["gc_content"])
gc_df.head()

In [ ]:
# --- binned empirical hit probability (3% windows) ---
gc_df["gc_bin"] = pd.cut(gc_df["gc_content"], bins=range(0, 100, 3))
bins = (
    gc_df.groupby("gc_bin", observed=True)
    .agg(n_genes=("gene_id", "count"), n_hits=("is_hit", "sum"))
    .assign(
        bin_mid=lambda d: d.index.map(lambda iv: iv.mid),
        p_hit=lambda d: d["n_hits"] / d["n_genes"],
    )
    .reset_index(drop=True)
    .query("n_genes >= 20")   # drop bins with very few genes
)

gc_df["sig"] = gc_df["is_hit"].apply(lambda x: "Sig. gene" if x == 1 else "Non-sig. gene")

print(f"Genes with GC content: {len(gc_df):,}")
print(f"  Hits:    {gc_df['is_hit'].sum():,}  ({gc_df['is_hit'].mean():.3%})")
print(f"  Not Hit: {(gc_df['is_hit'] == 0).sum():,}")
print(f"GC range: {gc_df['gc_content'].min():.1f}–{gc_df['gc_content'].max():.1f}%")
print(f"\n{len(bins)} bins retained after n_genes ≥ 20 filter")

In [ ]:
from scipy.stats import mannwhitneyu

# --- quadratic logistic regression ---
model_gc = smf.logit("is_hit ~ gc_content + I(gc_content**2)", data=gc_df).fit()
print(model_gc.summary())

# --- Mann–Whitney U: are GC distributions different between Hits and Not Hits? ---
hits_gc    = gc_df.loc[gc_df["is_hit"] == 1, "gc_content"]
nothits_gc = gc_df.loc[gc_df["is_hit"] == 0, "gc_content"]
mwu_stat, mwu_p = mannwhitneyu(hits_gc, nothits_gc, alternative="two-sided")

print(f"\nMann–Whitney U:  stat = {mwu_stat:.0f},  p = {mwu_p:.3g}")
print(f"Median GC — Hits: {hits_gc.median():.1f}%,  Not Hit: {nothits_gc.median():.1f}%")

In [ ]:
# --- quadratic logistic regression ---
model_gc_species = smf.logit("is_hit ~ species_name + gc_content + I(gc_content**2)", data=gc_df).fit()
print(model_gc_species.summary())

### Visualization

Two-panel figure:
- **Top**: fitted quadratic logistic curve with 95% confidence interval, overlaid on the empirical binned hit probabilities. Each point represents a 3% GC bin; marker area is proportional to the number of genes in that bin. The dashed horizontal line shows the baseline (overall) hit rate.
- **Bottom**: kernel density estimate of GC content separately for Hits and Not Hits, showing where each category is concentrated across the GC spectrum.

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

# --- fitted curve over observed GC range ---
gc_range = np.linspace(gc_df["gc_content"].min(), gc_df["gc_content"].max(), 400)
pred_df  = pd.DataFrame({"gc_content": gc_range})
pred     = model_gc.get_prediction(pred_df)
pred_mean = pred.predicted
pred_ci   = pred.conf_int(alpha=0.05)   # shape (n, 2): [:, 0] lower, [:, 1] upper
baseline = gc_df["is_hit"].mean()


## Figure 3e — GC content vs. hit probability

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(12, 4),
    gridspec_kw={"width_ratios": [1, 1.5], "wspace": 0.3},
)


# ── Panel 1: marginal KDE ────────────────────────────────────────────────────
ax = axes[0]
for call, color in [("Sig. gene", "#C0392B"), ("Non-sig. gene", "black")]:
    sub = gc_df[gc_df["sig"] == call]["gc_content"]
    sns.kdeplot(sub, ax=ax, color=color, fill=True, alpha=0.25, label=call)

ax.set_xlabel("GC content (%)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("GC content distribution by gene call", fontsize=12)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(10, 90)

# ── Panel 2: probability curve ───────────────────────────────────────────────
ax2 = axes[1]

ax2.fill_between(gc_range, pred_ci[:, 0], pred_ci[:, 1],
                alpha=0.2, color="#4472C4")
ax2.plot(gc_range, pred_mean, color="#4472C4", linewidth=2,
        label="Quadratic logistic fit")
ax2.axhline(baseline, color="gray", linestyle=":", linewidth=1,
           label=f"Baseline ({baseline:.3%})")

# Binned empirical probabilities — marker size ∝ n_genes
ax2.scatter(
    bins["bin_mid"], bins["p_hit"],
    s=bins["n_genes"] / 20,
    color="black", zorder=5, alpha=0.8,
    label="Empirical (3% bins)",
)

# Curve/line legend — upper left
legend1 = ax2.legend(fontsize=9, loc="upper left")
ax2.add_artist(legend1)

# Size legend — upper right
size_handles = [
    ax2.scatter([], [], s=n / 20, color="black", alpha=0.8, label=f"{n:,}")
    for n in [100, 500, 2000, 4000]
]
ax2.legend(
    handles=size_handles,
    title="Genes in bin",
    title_fontsize=8,
    fontsize=8,
    loc="upper right",
)

ax2.set_xlabel("GC content (%)", fontsize=12)
ax2.set_ylabel("Rate of sig. positive genes", fontsize=12)
ax2.set_title("Probability of being a Sig. gene by GC content", fontsize=13)
ax2.axvline(x=median_e_coli_gc, color="gray", linestyle="--", linewidth=1, label="Median E. coli GC content")
ax2.spines[["top", "right"]].set_visible(False)
#ax2.set_xlim(10, 90)

fig.savefig(RESULTS_DIR / "figure3e_gc_content_vs_hit_probability.pdf", bbox_inches="tight")
plt.show()


## Figure 3f — RBS binding score vs. hit probability

In [ ]:
rbs_df = per_gene_df.dropna(subset=["Sapiens_RBS_Score"]).copy()

# --- binned empirical hit probability (0.02-unit windows) ---
rbs_bin_edges = np.arange(0.0, 1.0, 0.05)
rbs_df["rbs_bin"] = pd.cut(rbs_df["Sapiens_RBS_Score"], bins=rbs_bin_edges)
rbs_bins = (
    rbs_df.groupby("rbs_bin", observed=True)
    .agg(n_genes=("gene_id", "count"), n_hits=("is_hit", "sum"))
    .assign(
        bin_mid=lambda d: d.index.map(lambda iv: iv.mid),
        p_hit=lambda d: d["n_hits"] / d["n_genes"],
    )
    .reset_index(drop=True)
    .query("n_genes >= 20")
)

# --- linear logistic regression ---
model_rbs = smf.logit("is_hit ~ Sapiens_RBS_Score", data=rbs_df).fit()
print(model_rbs.summary())

# --- Mann–Whitney U ---
hits_rbs    = rbs_df.loc[rbs_df["is_hit"] == 1, "Sapiens_RBS_Score"]
nothits_rbs = rbs_df.loc[rbs_df["is_hit"] == 0, "Sapiens_RBS_Score"]
mwu_stat_rbs, mwu_p_rbs = mannwhitneyu(hits_rbs, nothits_rbs, alternative="two-sided")
print(f"\nMann–Whitney U:  stat = {mwu_stat_rbs:.0f},  p = {mwu_p_rbs:.3g}")
print(f"Median RBS — Hits: {hits_rbs.median():.4f},  Not Hit: {nothits_rbs.median():.4f}")

# --- predictions over observed CAI range ---
rbs_range  = np.linspace(rbs_df["Sapiens_RBS_Score"].min(), rbs_df["Sapiens_RBS_Score"].max(), 400)
pred_rbs   = model_rbs.get_prediction(pd.DataFrame({"Sapiens_RBS_Score": rbs_range}))
pred_rbs_mean = pred_rbs.predicted
pred_rbs_ci   = pred_rbs.conf_int(alpha=0.05)

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(12, 4),
    gridspec_kw={"width_ratios": [1, 1.5], "wspace": 0.3},
)

# ── Panel 1: KDE of CAI for hits vs. non-hits ────────────────────────────────
ax_kde = axes[0]
for call, color in [("Hit", "#C0392B"), ("Not Hit", "black")]:
    sub = rbs_df[rbs_df["gene_call"] == call]["Sapiens_RBS_Score"]
    sns.kdeplot(sub, ax=ax_kde, color=color, fill=True, alpha=0.25, label=call)

ax_kde.set_xlabel("RBS binding score", fontsize=12)
ax_kde.set_ylabel("Density", fontsize=12)
ax_kde.set_title("RBS binding score distribution by gene call", fontsize=12)
ax_kde.legend(fontsize=9)
ax_kde.text(
    0.97, 0.95, f"Mann–Whitney U\np = {mwu_p_rbs:.3g}",
    transform=ax_kde.transAxes, ha="right", va="top", fontsize=10,
)
ax_kde.spines[["top", "right"]].set_visible(False)

# ── Panel 2: logistic regression probability curve + binned scatter ───────────
ax_lr = axes[1]
baseline_rbs = rbs_df["is_hit"].mean()

ax_lr.fill_between(
    rbs_range, pred_rbs_ci[:, 0], pred_rbs_ci[:, 1],
    color="#0B6623", alpha=0.20,
)
ax_lr.plot(
    rbs_range, pred_rbs_mean,
    color="#0B6623", linewidth=2, label="Linear logistic fit",
)
ax_lr.axhline(baseline_rbs, color="gray", linestyle=":", linewidth=1,
              label=f"Baseline ({baseline_rbs:.3%})")

ax_lr.scatter(
    rbs_bins["bin_mid"], rbs_bins["p_hit"],
    s=rbs_bins["n_genes"] / 20,
    color="black", zorder=4, alpha=0.8,
    label="Empirical (0.05 bins)",
)

# curve legend — upper left
legend1 = ax_lr.legend(fontsize=9, loc="upper left")
ax_lr.add_artist(legend1)

# size legend — upper right
size_handles = [
    ax_lr.scatter([], [], s=n / 20, color="black", alpha=0.8, label=f"{n:,}")
    for n in [100, 500, 2000, 4000]
]
ax_lr.legend(
    handles=size_handles,
    title="Genes in bin", title_fontsize=8,
    fontsize=8, loc="upper right", bbox_to_anchor=(0.85, 1.0),
)

ax_lr.set_xlabel("Predicted RBS affinity", fontsize=12)
ax_lr.set_ylabel("Rate of sig. positive genes", fontsize=12)
ax_lr.set_title("Probability of being a hit by RBS binding score", fontsize=13)
ax_lr.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR / "figure3f_rbs_vs_hit_probability.pdf", bbox_inches="tight")


In [ ]:
model_two_var = smf.logit("is_hit ~ Sapiens_RBS_Score + gc_content + I(gc_content**2)", data=per_gene_df).fit()
print(model_two_var.summary())

In [ ]:
stats.chi2.sf(-2* (model_gc.llf - model_two_var.llf), 1)

In [ ]:
model_three_var = smf.logit("is_hit ~ Sapiens_RBS_Score + gc_content_diff_ecoli + I(gc_content_diff_ecoli**2) + CAI_rel_to_Ecoli", data=per_gene_df).fit()
print(model_three_var.summary())

In [ ]:
stats.chi2.sf(-2* (model_two_var.llf - model_three_var.llf), 1)